# Find points where Sea Ice gets stuck due to a landmask constraint


In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test4-d28e0359/datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import os
import xarray as xr
import numpy as np
import cf_xarray
from datetime import timedelta
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client

import pandas as pd
pd.set_option('display.max_rows', 500)

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

In [ ]:
from dask.distributed import Client

In [ ]:
client = Client(threads_per_worker = 1)

In [ ]:
client.dashboard_link

### Load ACCESS-OM3 sea ice data

We only care about ice that gets stuck long term, lets average the last five years 

In [ ]:
hi_search = datastore.search(variable = "hi_m", realm='seaIce', frequency="1mon")

In [ ]:
ds = hi_search.to_dask()

In [ ]:
ds['hi_m'].max(['ni','nj']).plot()
plt.title('Maximum sea ice thickness')

In [ ]:
#Lets do the last 5 years only
ds=ds.isel(time=slice(-60,None)).load().mean('time')

In [ ]:
ds

In [ ]:
ds.load()

Also load depths, we need these to update the hand edits file

Youll have to set this manually

In [ ]:
topog_ds = xr.open_dataset('/g/data/vk83/configurations/inputs/access-om3/share/grids/global.25km/2026.06.11/topog.nc')

In [ ]:
topog_ds

# Find points with thick ice and plot them

In [ ]:
stacked = ds['hi_m'].stack(new_dim=('nj','ni'))

In [ ]:
stacked = stacked.where((stacked>6).compute(), drop=True)

In [ ]:
thick_ice = stacked.sortby(stacked, ascending=False)

Print some thicknii of the points found

In [ ]:
thick_ice.values

In [ ]:
ds

In [ ]:
buffer = 10 #cells
for point in thick_ice:
    plt.figure(figsize=(10,4))
    (y_i, x_i) = (point.nj.values, point.ni.values)
    plt.subplot(1,2,1)
    ds['hi_m'].sel(
        nj = slice(y_i-buffer,y_i+buffer),
        ni = slice(x_i-buffer,x_i+buffer)
    ).plot()
    plt.subplot(1,2,2)
    topog_ds['depth'].sel(
        ny = slice(y_i-buffer,y_i+buffer),
        nx = slice(x_i-buffer,x_i+buffer)
    ).plot()
    print(f"  {x_i}  {y_i}  {topog_ds.depth.isel(ny=y_i, nx=x_i).values}  0.0")
    plt.show()